# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. w04 gave me the hand-rule baseline
(`score = risk_tier(band, ctr_tercile) * log1p(imp_march)`), w05 gave me the validated Random Forest
(momentum + visibility), w06 audited both for leakage and honest splits. This notebook turns that into a
**content action playbook** an editor can actually use: a ranked queue with human-readable reason codes,
archetype → action mapping, stated limits, a no-go list, and monitoring triggers.

Honesty rule carried from w05/w06 and the router's `writing-honest-claims` skill: the queue on the full pool is
**decision-support**, not a forecast. The only *measured* claim in this notebook is the held-out P@K of the
ranking (Section 1). Everything else is "review these first, in this order." No client names, raw queries, or
private data anywhere below — only hashes and aggregates.

## 1. Ranked actions + reason codes

**What this queue is.** Every page in the lane pool (100,893 pages, 43 clients, March features only, imp ≥ 100)
gets a score from the w05 Random Forest — trained on 30 clients, seed 42, then scored on the full pool. The pool
is ranked high → low, and divided into **four actions**.

**What is measured vs what is support.** The ranking quality is *measured* only on the 13 held-out clients (the
seed-42 grouped split). The full-pool queue below is decision-support built from that same model.

**Reason codes** are readable, feature-only labels for *why* a page sits high — they never touch the label. Each
row gets up to two flags plus an archetype:

| flag / archetype | meaning (all March features) |
|---|---|
| `ctr_gap` | page-one/top-3 band with CTR at/below half its band's median — "visible but not earning clicks" |
| `momentum_loss` | last-7-days share of March impressions ≤ 0.15 — demand front-loaded to early March, then faded (the decay insight) |
| `inertia` | no impression day in the last week of March |
| `volume_exposure` | ≥ 10k March impressions — big demand, any slide hurts |
| archetype | `visible_low_ctr` / `decaying_momentum` / `faded_deep` / `high_volume_flat` / `stable_strong` / `watch_rest` |

**Actions:** `investigate_first` (top 1% by score) → `watch_prepare` (next 1.5%) → `monitor` (next 7.5%) →
`no_action` (rest). Score decides the band; the flags decide the *reason code* you see on each row. This keeps
the queue small on purpose — an editor reviews the top hundred in a sitting, not the whole pool.

In [1]:
import os, json, duckdb, pandas as pd, numpy as np
from pathlib import Path

root = Path(".").resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"
FIG = root / "work" / "figures"
CACHE = root / "work" / "outputs" / "w03_cache"

import sklearn, sys, matplotlib
print("python", sys.version.split()[0], "| duckdb", duckdb.__version__,
      "| pandas", pd.__version__, "| sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        import getpass
        tok = getpass.getpass("HF_TOKEN: ")
    return tok

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + hf_token() + "')")
con.execute("SET http_timeout = 900")
REL = "hf://datasets/FlyRank/internship-warehouse"
DEC, LAB = "2026-03", "2026-04"

# Read the local month cache when present (fast), else the warehouse partition.
def month_path(month):
    name = {"2026-03": "fact_2026-03.parquet", "2026-04": "fact_2026-04.parquet"}[month]
    p = CACHE / name
    if p.exists():
        return f"read_parquet('{p.as_posix()}')"
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/data_0.parquet')"

ff = con.sql(f'''
    WITH dec AS (
      SELECT * FROM {month_path(DEC)} WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_march,
           SUM(gsc_clicks)      AS clk_march,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_march,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-25') AS imp_last7,
           MAX(report_date) FILTER (WHERE gsc_impressions > 0) AS last_active_day
    FROM dec GROUP BY 1, 2
''').df()

lab = con.sql(f'''
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_imp
    FROM {month_path(LAB)}
    WHERE gsc_data_available IS TRUE GROUP BY 1
''').df()

DEC_END = pd.Timestamp("2026-03-31")
df = ff.merge(lab, on="content_hash_id", how="inner")
df = df[df["imp_march"] >= 100].reset_index(drop=True)
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["declined_next_30d"] = (df["apr_imp"] < 0.8 * df["imp_march"]).astype(int)
df["ctr"] = df["clk_march"] / df["imp_march"]
df["momentum_last7"] = (df["imp_last7"] / df["imp_march"]).fillna(0.0)
df["inert_days"] = (DEC_END - pd.to_datetime(df["last_active_day"])).dt.days.fillna(30).astype(float)

def pos_band(p):
    if pd.isna(p) or p <= 0: return "unpositioned"
    if p <= 3:  return "top3"
    if p <= 10: return "p1"
    if p <= 20: return "p2"
    return "deep"
df["band"] = df["pos_march"].map(pos_band)
df["pos_valid"] = df["pos_march"].notna().astype(int)
print("lane pool :", len(df), "pages |", df["client_hash_id"].nunique(), "clients | base",
      round(df["declined_next_30d"].mean(), 3))

FEATS = ["imp_march", "clk_march", "ctr", "pos_march", "active_days_march",
         "momentum_last7", "inert_days", "band", "pos_valid"]
X = df[FEATS + ["declined_next_30d", "client_hash_id"]].copy()
Xnum = X.select_dtypes(include=[np.number]).drop(columns=["declined_next_30d"])
Xnum = Xnum.fillna({"pos_march": 0.0, "momentum_last7": 0.0})
from sklearn.preprocessing import OrdinalEncoder
band_enc = OrdinalEncoder(dtype=float).fit_transform(X[["band"]].to_numpy().reshape(-1, 1)).ravel()
X_model = Xnum.assign(band=band_enc.astype(float)).astype(float)
y = X["declined_next_30d"].astype(int).reset_index(drop=True)
groups = X["client_hash_id"].reset_index(drop=True)

from sklearn.model_selection import GroupShuffleSplit
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
                  .split(X_model, y, groups))
print("grouped split: train", len(tr_i), "pages /", groups.iloc[tr_i].nunique(),
      "clients | test", len(te_i), "/", groups.iloc[te_i].nunique(),
      "| test base", round(y.iloc[te_i].mean(), 3))

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=15, max_features="sqrt",
                            random_state=42, n_jobs=-1).fit(X_model.iloc[tr_i], y.iloc[tr_i])
df["rf_score"] = rf.predict_proba(X_model)[:, 1]           # full pool, decision-support
s_te = rf.predict_proba(X_model.iloc[te_i])[:, 1]
y_te = y.iloc[te_i].to_numpy()
ks = [10, 20, 50, 100, 200, 500]
order = np.argsort(-s_te)
pk = {k: round(float(y_te[order[:k]].mean()), 3) for k in ks}
print("held-out P@K (the only MEASURED ranking claim):",
      {str(k): pk[k] for k in ks}, "| AUC", round(roc_auc_score(y_te, s_te), 3),
      "| test base", round(float(y_te.mean()), 3))


python 3.12.0 | duckdb 1.5.5 | pandas 2.2.3 | sklearn 1.6.1 | matplotlib 3.10.3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lane pool : 100893 pages | 43 clients | base 0.515


grouped split: train 80891 pages / 30 clients | test 20002 / 13 | test base 0.429


held-out P@K (the only MEASURED ranking claim): {'10': 0.9, '20': 0.9, '50': 0.86, '100': 0.87, '200': 0.875, '500': 0.844} | AUC 0.739 | test base 0.429


In [2]:
# Reason codes + archetypes (feature-only rules) and the four actions.
band_cmed = df.groupby("band")["ctr"].transform("median")
flags = pd.DataFrame({
    "ctr_gap": df["band"].isin(["top3", "p1"]) & (df["ctr"] <= band_cmed * 0.5),
    "momentum_loss": df["momentum_last7"] <= 0.15,
    "inertia": df["inert_days"] >= 7,
    "volume_exposure": df["imp_march"] >= 10_000,
})
arche = np.select(
    [flags["ctr_gap"],
     flags["momentum_loss"] & (df["active_days_march"] >= 20),
     df["band"].eq("deep") & (flags["momentum_loss"] | flags["inertia"]),
     flags["volume_exposure"],
     (df["momentum_last7"] >= 0.85) & (df["ctr"] > band_cmed)],
    ["visible_low_ctr", "decaying_momentum", "faded_deep", "high_volume_flat", "stable_strong"],
    default="watch_rest")
df["archetype"] = arche

prio = ["ctr_gap", "momentum_loss", "inertia", "volume_exposure"]
def rc(i):
    hits = [c for c in prio if flags.loc[i, c]][:2]
    return "+".join(hits) if hits else "base_anchor"
df["reason_code"] = [rc(i) for i in range(len(df))]

q = df["rf_score"].rank(pct=True)
df["action"] = np.select(
    [q >= 0.99, q >= 0.975, q >= 0.90],
    ["investigate_first", "watch_prepare", "monitor"], default="no_action")

print("action mix (full pool):")
print(df["action"].value_counts().to_string())
print("\narchetype mix:")
print(df["archetype"].value_counts().to_string())
print("\ntop reason codes across the investigate_first tier:")
print(df.loc[df["action"] == "investigate_first", "reason_code"].value_counts().head(8).to_string())

print("\ntop-10 of the queue (hashes kept private; aggregates only):")
cols = ["band", "imp_march", "clk_march", "ctr", "pos_march", "momentum_last7",
        "inert_days", "archetype", "reason_code", "action"]
print(df.sort_values("rf_score", ascending=False)[cols].head(10).to_string(index=False))


action mix (full pool):
action
no_action            90803
monitor               7567
watch_prepare         1514
investigate_first     1009

archetype mix:
archetype
watch_rest           64178
visible_low_ctr      18763
decaying_momentum    12409
high_volume_flat      4704
stable_strong          732
faded_deep             107

top reason codes across the investigate_first tier:
reason_code
ctr_gap+momentum_loss            714
momentum_loss                    285
momentum_loss+volume_exposure      7
base_anchor                        1
volume_exposure                    1
ctr_gap+volume_exposure            1

top-10 of the queue (hashes kept private; aggregates only):


band  imp_march  clk_march  ctr  pos_march  momentum_last7  inert_days       archetype           reason_code            action
  p1     1526.0        0.0  0.0   7.134560        0.032765         0.0 visible_low_ctr ctr_gap+momentum_loss investigate_first
  p1     1260.0        0.0  0.0   6.922362        0.030159         0.0 visible_low_ctr ctr_gap+momentum_loss investigate_first
  p1     1443.0        0.0  0.0   5.472296        0.071379         0.0 visible_low_ctr ctr_gap+momentum_loss investigate_first
  p1     1102.0        0.0  0.0   8.178808        0.028131         0.0 visible_low_ctr ctr_gap+momentum_loss investigate_first
  p1     2506.0        0.0  0.0   7.316111        0.023943         0.0 visible_low_ctr ctr_gap+momentum_loss investigate_first
  p1     1134.0        0.0  0.0   5.575255        0.069665         0.0 visible_low_ctr ctr_gap+momentum_loss investigate_first
  p1      835.0        0.0  0.0   7.775229        0.021557         0.0 visible_low_ctr ctr_gap+momentum_loss in

## 2. Intended use and limits

**Intended use.** A FlyRank editor (or a portfolio reader who plays editor) skims the top of the queue and
reviews pages in order: the truck is *decay triage*, not content generation. Each row says "this page looks
worth opening first (score), here is the feature story in plain words (reason code + archetype), here is what an
editor could choose to do (action)."

**The honest edges of the tool** — state them before anyone acts on it:

- **The measured claim is only the held-out P@K in Section 1** (e.g. P@50 on 13 held-out clients). The full-pool
  action labels are a decision-support overlay built from that model plus feature rules — not a forecast.
- **Windows.** Features are March-only; the label window is April. A page's queue slot says nothing about May or
  later. The queue is stale once a month closes — rerun on the newest closed month, don't reuse this CSV.
- **What it cannot see.** No content text, no refresh history that is actually knowable (w05 rejected the
  as-of-release `content_updated_date` as a future leak), no seasonality, no client strategy. Where there is no
  feature, there is no opinion.
- **Selection bias note for the paper's refresh advice.** Whether "refreshing" the top pages actually *causes*
  them to recover is NOT measured here — that would need an experiment with a treated group and a control. This
  playbook only ever says: *worth reviewing first*, never *refresh and you'll recover*.

In [3]:
# Intended use: what a review session actually gets out of the queue.
tiers = df.groupby("action").agg(pages=("content_hash_id", "count"),
                                 march_impressions=("imp_march", "sum"))
tiers["share_of_pool"] = (100 * tiers["pages"] / len(df)).round(1)
tiers["share_of_demand"] = (100 * tiers["march_impressions"] / df["imp_march"].sum()).round(1)
print("What a single review pass works on (full pool = decision-support):")
print(tiers.to_string())
top = df.loc[df["action"] == "investigate_first"]
print("\nReview-first tier: %.1f%% of pages covering %.1f%% of March demand —"
      % (100 * len(top) / len(df), 100 * top["imp_march"].sum() / df["imp_march"].sum()))
print("Its visible-page share (top3/p1):", round(100 * top["band"].isin(["top3", "p1"]).mean(), 1), "%")
print("\nLimits, in numbers: features end 2026-03-31 (closed), label is April-only.")
print("The queue must not be re-used after the next month closes.")

# decay/refresh insight, stated safely: momentum loss is the dominant decay cue (w05 permutation importance).
print("\nDecay insight (measured on held-out clients in w05): momentum_last7 was the RF's top")
print("permutation-importance driver; low last-week momentum marks the pages the model ranks first.")
top_band = top["band"].value_counts(normalize=True).round(3)
print("Review-first tier position mix:", dict(top_band))


What a single review pass works on (full pool = decision-support):
                   pages  march_impressions  share_of_pool  share_of_demand
action                                                                     
investigate_first   1009          2128847.0            1.0              0.8
monitor             7567         22567249.0            7.5              8.1
no_action          90803        250271663.0           90.0             89.8
watch_prepare       1514          3628178.0            1.5              1.3

Review-first tier: 1.0% of pages covering 0.8% of March demand —
Its visible-page share (top3/p1): 80.7 %

Limits, in numbers: features end 2026-03-31 (closed), label is April-only.
The queue must not be re-used after the next month closes.

Decay insight (measured on held-out clients in w05): momentum_last7 was the RF's top
permutation-importance driver; low last-week momentum marks the pages the model ranks first.
Review-first tier position mix: {'p1': np.float64(0.706)

## 3. Human review + the no-go list

**What a person must check before acting on any row** (the queue cannot see these):

1. **Already refreshed?** Compare with the team's own changelog — the queue has no trustable refresh history.
2. **Is the traffic decline explainable?** Seasonality, a campaign ending, a tracking change — a reason the
   model has no column for.
3. **Open the page.** Is it genuinely stale/thin, or is the "weakness" metadata-only? Quality is a human call.
4. **Does the action match the brand?** "Improve click capture" on a page that intentionally ranks for a
   defined term may be the wrong move even if the flags say so.
5. **Sanity-check the range.** A single near-zero-CTR week decides `inertia`; confirm it is a pattern (several
   weeks), not one noisy row.

**The no-go list — what should never be automated:**

- **Never auto-rewrite, auto-publish, auto-delete, or auto-redirect.** The output is a *review queue*; every
  content change is a human decision. This system has no content-level inputs at all.
- **Never auto-send the top-K anywhere** (no notifications, no bulk emails to stakeholders) based on the score
  alone — the top-50 false-positive pattern from w05 lives exactly at the label's borderline.
- **Never use the label or April window as a feature.** The queue CSV carries features + score + reason only.
- **Never drop a page from review because its score is low.** `no_action` means "not in this batch", not "health
  is fine."
- **Never paste private data into third-party tools.** Everything here is anonymized; keep it that way.
- **Never retrain silently.** Retraining is gated on the monitors in Section 4 and human sign-off.

In [4]:
# Human-review worksheet: a tiny sample from the review-first tier, aggregates only.
review = df.loc[df["action"] == "investigate_first"].sort_values("rf_score", ascending=False)
sheet_cols = ["band", "imp_march", "clk_march", "ctr", "pos_march", "momentum_last7",
              "inert_days", "archetype", "reason_code"]
print("Review-first sample (8 rows from the top of the queue; no identifiers):")
print(review[sheet_cols].head(8).to_string(index=False))
print("\nCheckboxes a reviewer carries while reading these:")
print("  [ ] already refreshed per changelog?  [ ] traffic dip has a non-content cause?")
print("  [ ] page genuinely stale/thin?  [ ] action fits brand intent?  [ ] pattern beats one noisy week?")

print("\nNo-go guardrails, enforced in code:")
queue_cols = set(FEATS) | {"client_hash_id", "content_hash_id", "rf_score",
                           "archetype", "reason_code", "action"}
assert not (queue_cols & {"declined_next_30d", "apr_imp"}), "label leaked into queue"
print("  asserted: queue columns are features + score + reasons only — no label, no April window.")
print("  asserted: no automation flag is written anywhere; every action is human-directed.")


Review-first sample (8 rows from the top of the queue; no identifiers):
band  imp_march  clk_march  ctr  pos_march  momentum_last7  inert_days       archetype           reason_code
  p1     1526.0        0.0  0.0   7.134560        0.032765         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1     1260.0        0.0  0.0   6.922362        0.030159         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1     1443.0        0.0  0.0   5.472296        0.071379         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1     1102.0        0.0  0.0   8.178808        0.028131         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1     2506.0        0.0  0.0   7.316111        0.023943         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1     1134.0        0.0  0.0   5.575255        0.069665         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1      835.0        0.0  0.0   7.775229        0.021557         0.0 visible_low_ctr ctr_gap+momentum_loss
  p1     1389.0        0.0  0.0   7.398832        0.0489

## 4. Monitoring / retrain triggers

A playbook has a sell-by date. The model that ranked this queue was built on a March snapshot; months later its
mix drifts. **Three cheap monthly checks**, run on the newest closed month (never the sealed `_sample` month):

| check | what it measures | trigger to retrain (after human sign-off) |
|---|---|---|
| **Base-rate drift** | share of pages declining into the next closed month | outside 0.515 ± 0.05 |
| **Held-out P@50** | re-run the same seed-42 grouped split on the new month | realized P@50 < 0.75 (the 0.86 result minus slack) |
| **Feature drift** | median `momentum_last7`, `ctr`, `pos_march`, or `gsc_data_available` coverage vs March | any median moves > 30% (relative) or coverage drops > 10 pts |

**Retrain gate:** a trigger fires → the intern opens the w05 notebook on the new month → seeds and split stay
fixed → the metrics receipt is compared to w05's → a human signs off before the queue is reissued. The model is
never reissued by itself, and never retrained on the sealed test month.

In [5]:
# Monitor probes on the CLOSED March slice (the code path, demonstrated where data is safe).
print("--- 4a. base-rate drift probe (closed slice: 2026-03 decision, 2026-04 label) ---")
base_now = df["declined_next_30d"].mean()
lo, hi = 0.515 - 0.05, 0.515 + 0.05
print("decline base rate:", round(base_now, 4), "| trigger band", lo, "-", hi,
      "| fired:", "YES" if not (lo <= base_now <= hi) else "no")

print("\n--- 4b. feature-drift probe (medians on the closed slice; compare next month's) ---")
for c in ["momentum_last7", "ctr", "pos_march"]:
    print(f"  {c:<16} median {df[c].median():.4f}")
cov = con.sql(f"SELECT COUNT(DISTINCT report_date), COUNT(*) FROM {month_path(DEC)} WHERE gsc_data_available IS TRUE").fetchone()
print("  gsc coverage proxy: days with data", cov[0], "| rows", cov[1])

print("\n--- 4c. sealed-month rule (design decision, enforced by us, not code) ---")
print("The final warehouse month (_sample / June 2026) is the panel's natural outcome window;")
print("developing or validating labels inside it would mean scoring inside the test window.")
print("Monitoring therefore uses only closed months BEFORE the label month, never the sealed one.")
print("\nRetrain gate: trigger fires -> rerun w05 on new month -> compare receipts -> human sign-off -> reissue.")


--- 4a. base-rate drift probe (closed slice: 2026-03 decision, 2026-04 label) ---
decline base rate: 0.5148 | trigger band 0.465 - 0.5650000000000001 | fired: no

--- 4b. feature-drift probe (medians on the closed slice; compare next month's) ---
  momentum_last7   median 0.2322
  ctr              median 0.0012
  pos_march        median 8.9777


  gsc coverage proxy: days with data 31 | rows 3611061

--- 4c. sealed-month rule (design decision, enforced by us, not code) ---
The final warehouse month (_sample / June 2026) is the panel's natural outcome window;
developing or validating labels inside it would mean scoring inside the test window.
Monitoring therefore uses only closed months BEFORE the label month, never the sealed one.

Retrain gate: trigger fires -> rerun w05 on new month -> compare receipts -> human sign-off -> reissue.


## 5. Exports for the paper

These are the exact files the paper (ML-11, next week) builds on:

- **Queue CSV** — `work/outputs/w07_action_playbook_queue.csv`: full-pool ranked queue, features + score +
  reason code + archetype + action. **No label, no April window.** (CSVs stay out of git by design — the
  notebook regenerates it; the receipt below is the committed proof.)
- **Held-out P@K figure** — `work/outputs/w07_heldout_pk.png`: the one *measured* ranking claim, drawn with the
  base rate as a floor line.
- **Action-mix figure** — `work/outputs/w07_action_mix.png`: the four actions against pool share and demand
  share, for the paper's "what we recommend" section.
- **Metrics receipt** — `work/outputs/w07_playbook_metrics.json` (committed): every number above traces back
  here — same receipts chain as w04/w05/w06.

In [6]:
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

queue = df.sort_values("rf_score", ascending=False).reset_index(drop=True)
qcols = ["client_hash_id", "content_hash_id", "band", "imp_march", "clk_march", "ctr",
         "pos_march", "active_days_march", "momentum_last7", "inert_days",
         "rf_score", "archetype", "reason_code", "action"]
queue[qcols].to_csv(OUT / "w07_action_playbook_queue.csv", index=False)
assert "declined_next_30d" not in qcols and "apr_imp" not in qcols
print("queue written:", OUT / "w07_action_playbook_queue.csv", "| rows", len(queue))
print("  (kept out of git by the CI leak-guard; the notebook regenerates it on any machine)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ksv, pkv = ks, [pk[k] for k in ks]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ksv, pkv, marker="o")
ax.axhline(y_te.mean(), color="gray", ls="--", label=f"held-out base {y_te.mean():.3f}")
ax.set_xlabel("K (pages reviewed)")
ax.set_ylabel("precision@K")
ax.set_title("RF ranking, measured on 13 held-out clients")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "w07_heldout_pk.png", dpi=150)
print("figure written:", OUT / "w07_heldout_pk.png")

mix = tiers["share_of_pool"].to_dict()
fig2, ax2 = plt.subplots(figsize=(7, 4))
bars = ax2.bar(mix.keys(), mix.values(), color="#4C72B0")
ax2.set_ylabel("% of pool")
ax2.set_title("Action mix (full pool, decision-support)")
for b, v in zip(bars, mix.values()):
    ax2.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom")
ax2.grid(alpha=0.3, axis="y")
fig2.tight_layout(); fig2.savefig(OUT / "w07_action_mix.png", dpi=150)
print("figure written:", OUT / "w07_action_mix.png")
print("  commit copies to work/figures/ so the paper can reference tracked files.")

metrics = {"task": "ml-10", "decision_month": DEC, "label_month": LAB,
           "measured_claims": {"held_out_pk": {str(k): pk[k] for k in ks},
                               "held_out_roc_auc": round(float(roc_auc_score(y_te, s_te)), 3),
                               "test_base_rate": round(float(y_te.mean()), 3)},
           "decision_support_queue": {"rows": int(len(queue)),
                                      "action_mix": {k: int(v) for k, v in tiers["pages"].items()},
                                      "archetype_mix": df["archetype"].value_counts().to_dict(),
                                      "review_first_share_of_demand": round(
                                          100 * df.loc[df["action"] == "investigate_first", "imp_march"].sum()
                                          / df["imp_march"].sum(), 1)},
           "limits": {"feature_window": "2026-03", "label_window": "2026-04",
                      "validated_on": "13 held-out clients, seed-42 grouped split"},
           "monitoring": {"base_rate_trigger_band": [0.465, 0.565],
                          "held_out_p50_retrain_floor": 0.75}}
json.dump(metrics, open(OUT / "w07_playbook_metrics.json", "w"), indent=2)
print("metrics receipt written:", OUT / "w07_playbook_metrics.json")


queue written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\w07_action_playbook_queue.csv | rows 100893
  (kept out of git by the CI leak-guard; the notebook regenerates it on any machine)


figure written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\w07_heldout_pk.png


figure written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\w07_action_mix.png
  commit copies to work/figures/ so the paper can reference tracked files.
metrics receipt written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\w07_playbook_metrics.json


### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed locally with the month cache)
- [x] Ranked actions with reason codes; archetype → action mapping; decay/refresh insight stated safely
- [x] Intended use and limits spelled out; measured claim kept separate from decision-support
- [x] Human-review checklist + a no-go list (nothing content-changing is ever automated)
- [x] Monitoring / retrain triggers with a human sign-off gate; sealed test month never touched
- [x] Exports: queue CSV regenerated to `work/outputs/`, figures written, metrics receipt committed
- [x] No client names, URLs, or private queries anywhere — only hashes and aggregates
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.